# 01 - Carga del dataset WGISD

Descargar WGISD y explorar su estructura: imagenes, cajas YOLO, mascaras `.npz`,
variedad y color por nombre de archivo.

A verificar aca:
- 300 imagenes, 2048px de ancho.
- Split oficial: train 242 imagenes / test 58.
- Mascara `M[:,:,i]` corresponde a la caja `i` del archivo de cajas.

In [1]:
import sys
from collections import Counter
from pathlib import Path

import numpy as np
from PIL import Image

sys.path.append(str(Path.cwd().parent / "scripts"))

from download_wgisd import download

WGISD_DIR = Path("../data/wgisd")
download(WGISD_DIR)

../data/wgisd ya existe y no está vacío, no se vuelve a clonar.


In [2]:
# estructura del repo clonado
for p in sorted(WGISD_DIR.iterdir()):
    print(p.name)

.git
CODE_OF_CONDUCT.md
LICENSE
README.md
WGISD.ipynb
classes.txt
coco_annotations
contrib
data
extra
original_resolution
test.txt
test_berries.txt
test_masked.txt
train.txt
train_berries.txt
train_masked.txt
val_berries.txt


## Listar imagenes y clasificar por variedad/color

Variedad en el prefijo del nombre: CDY (Chardonnay), CFR (Cabernet Franc),
CSV (Cabernet Sauvignon), SVB (Sauvignon Blanc), SYH (Syrah).
Blancas: CDY, SVB. Tintas: CFR, CSV, SYH.

In [3]:
VARIETY_COLOR = {
    "CDY": "blanca",
    "SVB": "blanca",
    "CFR": "tinta",
    "CSV": "tinta",
    "SYH": "tinta",
}

IMAGES_DIR = WGISD_DIR / "data"
image_paths = sorted(IMAGES_DIR.glob("*.jpg"))
print("cantidad de imagenes:", len(image_paths))

variety_counts = Counter(p.name[:3] for p in image_paths)
print(variety_counts)

cantidad de imagenes: 300
Counter({'CDY': 65, 'CFR': 65, 'SVB': 65, 'CSV': 57, 'SYH': 48})


## Cargar cajas YOLO y mascaras npz de un ejemplo

In [4]:
example = image_paths[0]
img = Image.open(example)
print(example.name, img.size)

boxes_path = example.with_suffix(".txt")
boxes = np.loadtxt(boxes_path).reshape(-1, 5)  # class cx cy w h, normalizadas
print("cajas:", boxes.shape)

mask_path = example.with_suffix(".npz")
if mask_path.exists():
    npz = np.load(mask_path)
    M = npz["arr_0"] if "arr_0" in npz else npz[list(npz.keys())[0]]
    print("mascaras M shape (H, W, n_clusters):", M.shape)
    assert M.shape[2] == boxes.shape[0], "n_clusters deberia coincidir con n cajas"
else:
    print("esta imagen no tiene mascara")

CDY_2015.jpg (2048, 1365)
cajas: (24, 5)
mascaras M shape (H, W, n_clusters): (1365, 2048, 24)


## Split oficial

El repo WGISD trae listas `train.txt` / `test.txt` (nombres sin extension).

In [5]:
train_list = (WGISD_DIR / "train.txt").read_text().splitlines()
test_list = (WGISD_DIR / "test.txt").read_text().splitlines()
print("train:", len(train_list), "- test:", len(test_list))

def count_boxes_and_masks(names):
    n_boxes, n_masks_imgs, n_masks_boxes = 0, 0, 0
    for name in names:
        b = np.loadtxt(IMAGES_DIR / f"{name}.txt").reshape(-1, 5)
        n_boxes += b.shape[0]
        npz_p = IMAGES_DIR / f"{name}.npz"
        if npz_p.exists():
            n_masks_imgs += 1
            npz = np.load(npz_p)
            M = npz[list(npz.keys())[0]]
            n_masks_boxes += M.shape[2]
    return n_boxes, n_masks_imgs, n_masks_boxes

print("train - cajas, imagenes con mascara, racimos con mascara:", count_boxes_and_masks(train_list))
print("test  - cajas, imagenes con mascara, racimos con mascara:", count_boxes_and_masks(test_list))

train: 242 - test: 58
train - cajas, imagenes con mascara, racimos con mascara: (3580, 110, 1611)
test  - cajas, imagenes con mascara, racimos con mascara: (850, 27, 408)
